## All you need is love… And a pet!

<img src="img/dataset-cover.jpg" width="920">

Here we are going to build a classifier to predict whether an animal from an animal shelter will be adopted or not (aac_intakes_outcomes.csv, available at: https://www.kaggle.com/aaronschlegel/austin-animal-center-shelter-intakes-and-outcomes/version/1#aac_intakes_outcomes.csv). You will be working with the following features:

1. *animal_type:* Type of animal. May be one of 'cat', 'dog', 'bird', etc.
2. *intake_year:* Year of intake
3. *intake_condition:* The intake condition of the animal. Can be one of 'normal', 'injured', 'sick', etc.
4. *intake_number:* The intake number denoting the number of occurrences the animal has been brought into the shelter. Values higher than 1 indicate the animal has been taken into the shelter on more than one occasion.
5. *intake_type:* The type of intake, for example, 'stray', 'owner surrender', etc.
6. *sex_upon_intake:* The gender of the animal and if it has been spayed or neutered at the time of intake
7. *age_upon\_intake_(years):* The age of the animal upon intake represented in years
8. *time_in_shelter_days:* Numeric value denoting the number of days the animal remained at the shelter from intake to outcome.
9. *sex_upon_outcome:* The gender of the animal and if it has been spayed or neutered at time of outcome
10. *age_upon\_outcome_(years):* The age of the animal upon outcome represented in years
11. *outcome_type:* The outcome type. Can be one of ‘adopted’, ‘transferred’, etc.

In [8]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy as sp
from itertools import combinations 
import ast
from sklearn.linear_model import LogisticRegression
import seaborn as sn
%matplotlib inline

data_folder = './data/'

### A) Load the dataset and convert categorical features to a suitable numerical representation (use dummy-variable encoding). 
- Split the data into a training set (80%) and a test set (20%). Pair each feature vector with the corresponding label, i.e., whether the outcome_type is adoption or not. 
- Standardize the values of each feature in the data to have mean 0 and variance 1.

The use of external libraries is not permitted in part A, except for numpy and pandas. 
You can drop entries with missing values.

In [33]:
columns = ['animal_type', 'intake_year', 'intake_condition', 'intake_number', 'intake_type', 'sex_upon_intake', 'age_upon_intake_(years)', 'time_in_shelter_days', 'sex_upon_outcome', 'age_upon_outcome_(years)', 'outcome_type']
data = pd.read_csv(data_folder + 'aac_intakes_outcomes.csv', usecols = columns)

data.dropna(inplace= True)
data['adopted'] = (data['outcome_type'] == 'Adoption').astype(int)

data

,outcome_type,sex_upon_outcome,age_upon_outcome_(years),animal_type,intake_condition,intake_type,sex_upon_intake,age_upon_intake_(years),intake_year,intake_number,time_in_shelter_days,adopted
0,Return to Owner,Neutered Male,10.000000,Dog,Normal,Stray,Neutered Male,10.000000,2017,1.0,0.588194,0
1,Return to Owner,Neutered Male,7.000000,Dog,Normal,Public Assist,Neutered Male,7.000000,2014,2.0,1.259722,0
2,Return to Owner,Neutered Male,6.000000,Dog,Normal,Public Assist,Neutered Male,6.000000,2014,3.0,1.113889,0
3,Transfer,Neutered Male,10.000000,Dog,Normal,Owner Surrender,Neutered Male,10.000000,2014,1.0,4.970139,0
4,Return to Owner,Neutered Male,16.000000,Dog,Injured,Public Assist,Neutered Male,16.000000,2013,1.0,0.119444,0
...,...,...,...,...,...,...,...,...,...,...,...,...
79667,Transfer,Unknown,0.038356,Cat,Normal,Stray,Unknown,0.038356,2018,1.0,0.077083,0
79668,Euthanasia,Unknown,2.000000,Other,Normal,Wildlife,Unknown,2.000000,2018,1.0,0.053472,0
79669,Euthanasia,Unknown,1.000000,Other,Normal,Wildlife,Unknown,1.000000,2018,1.0,0.047917,0
79670,Return to Owner,Intact Male,0.821918,Dog,Normal,Stray,Intact Male,0.410959,2018,1.0,1.762500,0


In [ ]:
#Splitting 

training_split = 0.8

def splitting(set, split):
    train_size = int(split * set.shape[0])

    indexes = np.random.permutation(set.shape[0])

    train_indexes = indexes[:train_size]
    test_indexes = indexes[train_size:]

    train_set = set.iloc[train_indexes]
    test_set = set.iloc[test_indexes]

    return train_set, test_set


train, test = splitting(data, training_split)

In [ ]:
#Categorical and Dummies

categorical = ['sex_upon_outcome', 'animal_type', 'intake_condition', 'intake_type', 'sex_upon_intake']

train_categorical = pd.get_dummies(train, columns= categorical)

test_categorical = pd.get_dummies(test, columns= categorical)[train_categorical.columns]

,outcome_type,age_upon_outcome_(years),age_upon_intake_(years),intake_year,intake_number,time_in_shelter_days,adopted,sex_upon_outcome_Intact Female,sex_upon_outcome_Intact Male,sex_upon_outcome_Neutered Male,...,intake_type_Euthanasia Request,intake_type_Owner Surrender,intake_type_Public Assist,intake_type_Stray,intake_type_Wildlife,sex_upon_intake_Intact Female,sex_upon_intake_Intact Male,sex_upon_intake_Neutered Male,sex_upon_intake_Spayed Female,sex_upon_intake_Unknown
15491,Transfer,0.057534,0.057534,2014,1.0,0.034028,0,False,True,False,...,False,False,False,True,False,False,True,False,False,False
23805,Adoption,2.000000,2.000000,2014,1.0,63.130556,1,False,False,False,...,False,False,False,True,False,True,False,False,False,False
78334,Adoption,0.164384,0.164384,2018,1.0,3.935417,1,False,False,False,...,False,False,False,True,False,True,False,False,False,False
4556,Adoption,0.164384,0.082192,2013,1.0,5.275694,1,False,False,False,...,False,False,False,True,False,True,False,False,False,False
23605,Transfer,5.000000,5.000000,2014,1.0,3.786806,0,False,True,False,...,False,False,False,True,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
600,Return to Owner,3.000000,3.000000,2016,1.0,0.794444,0,False,True,False,...,False,False,False,True,False,False,True,False,False,False
24731,Transfer,8.000000,8.000000,2014,1.0,4.229861,0,True,False,False,...,False,False,False,True,False,True,False,False,False,False
65905,Transfer,2.000000,2.000000,2017,1.0,49.118056,0,False,False,True,...,False,True,False,False,False,False,False,True,False,False
65041,Return to Owner,2.000000,2.000000,2017,1.0,0.185417,0,False,True,False,...,False,False,False,True,False,False,True,False,False,False


In [56]:
train_label = train_categorical.adopted
train_features = train_categorical.drop(['adopted', 'outcome_type'], axis = 1)
test_label = test_categorical.adopted
test_features = test_categorical.drop(['adopted', 'outcome_type'], axis = 1)

display(train_features)

for c in train_features.columns:
    mean = train_features[c].mean()
    std = train_features[c].std()
    train_features[c] = (train_features[c] - mean) / std
    test_features[c] = (test_features[c] - mean) / std

,age_upon_outcome_(years),age_upon_intake_(years),intake_year,intake_number,time_in_shelter_days,sex_upon_outcome_Intact Female,sex_upon_outcome_Intact Male,sex_upon_outcome_Neutered Male,sex_upon_outcome_Spayed Female,sex_upon_outcome_Unknown,...,intake_type_Euthanasia Request,intake_type_Owner Surrender,intake_type_Public Assist,intake_type_Stray,intake_type_Wildlife,sex_upon_intake_Intact Female,sex_upon_intake_Intact Male,sex_upon_intake_Neutered Male,sex_upon_intake_Spayed Female,sex_upon_intake_Unknown
15491,0.057534,0.057534,2014,1.0,0.034028,False,True,False,False,False,...,False,False,False,True,False,False,True,False,False,False
23805,2.000000,2.000000,2014,1.0,63.130556,False,False,False,True,False,...,False,False,False,True,False,True,False,False,False,False
78334,0.164384,0.164384,2018,1.0,3.935417,False,False,False,True,False,...,False,False,False,True,False,True,False,False,False,False
4556,0.164384,0.082192,2013,1.0,5.275694,False,False,False,True,False,...,False,False,False,True,False,True,False,False,False,False
23605,5.000000,5.000000,2014,1.0,3.786806,False,True,False,False,False,...,False,False,False,True,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
600,3.000000,3.000000,2016,1.0,0.794444,False,True,False,False,False,...,False,False,False,True,False,False,True,False,False,False
24731,8.000000,8.000000,2014,1.0,4.229861,True,False,False,False,False,...,False,False,False,True,False,True,False,False,False,False
65905,2.000000,2.000000,2017,1.0,49.118056,False,False,True,False,False,...,False,True,False,False,False,False,False,True,False,False
65041,2.000000,2.000000,2017,1.0,0.185417,False,True,False,False,False,...,False,False,False,True,False,False,True,False,False,False


### B) Train a logistic regression classifier on your training set. Logistic regression returns probabilities as predictions, so in order to arrive at a binary prediction, you need to put a threshold on the predicted probabilities. 
- For the decision threshold of 0.5, present the performance of your classifier on the test set by displaying the confusion matrix. Based on the confusion matrix, manually calculate accuracy, precision, recall, and F1-score with respect to the positive and the negative class. 

In [63]:
def confusion_matrix (true_label, prediciton_proba, decision_threshold = 0.5):

    predicted_label = (prediciton_proba[:,1] > decision_threshold).astype(int)

    TP = np.sum(np.logical_and(predicted_label==1, true_label==1))
    TN = np.sum(np.logical_and(predicted_label==0, true_label==0))
    FP = np.sum(np.logical_and(predicted_label==1, true_label==0))
    FN = np.sum(np.logical_and(predicted_label==0, true_label==1))

    confusion_matrix = np.asarray([[TP, FP], [FN, TN]])

    confusion_matrix_float = confusion_matrix.astype(float)

    accuracy =  (TP+TN)/np.sum(confusion_matrix_float)

    precision_positive = TP/(TP+FP) if (TP+FP) !=0 else np.nan
    precision_negative = TN/(TN+FN) if (TN+FN) !=0 else np.nan
    
    recall_positive = TP/(TP+FN) if (TP+FN) !=0 else np.nan
    recall_negative = TN/(TN+FP) if (TN+FP) !=0 else np.nan

    F1_score_positive = 2 *(precision_positive*recall_positive)/(precision_positive+recall_positive) if (precision_positive+recall_positive) !=0 else np.nan
    F1_score_negative = 2 *(precision_negative*recall_negative)/(precision_negative+recall_negative) if (precision_negative+recall_negative) !=0 else np.nan

    return confusion_matrix, [decision_threshold, accuracy, precision_positive, recall_positive, F1_score_positive, precision_negative, recall_negative, F1_score_negative]

In [64]:
logistic = LogisticRegression(solver='lbfgs', max_iter=10000)
logistic.fit(train_features,train_label)

LogisticRegression(max_iter=10000)

In [65]:
predicted_proba = logistic.predict_proba(test_features)

In [68]:
[t, accuracy, precision_positive, recall_positive, F1_score_positive, precision_negative, recall_negative, F1_score_negative] = confusion_matrix(test_label, predicted_proba)[1]

print("The accuracy of this model is {0:1.3f}".format(accuracy))
print("For the positive case, the precision is {0:1.3f}, the recall is {1:1.3f} and the F1 score is {2:1.3f}"\
      .format(precision_positive, recall_positive, F1_score_positive))
print("For the negative case, the precision is {0:1.3f}, the recall is {1:1.3f} and the F1 score is {2:1.3f}"\
      .format(precision_negative, recall_negative, F1_score_negative))

The accuracy of this model is 0.824
For the positive case, the precision is 0.776, the recall is 0.821 and the F1 score is 0.798
For the negative case, the precision is 0.862, the recall is 0.826 and the F1 score is 0.844


### C) Vary the value of the threshold in the range from 0 to 1 and visualize the value of accuracy, precision, recall, and F1-score (with respect to both classes) as a function of the threshold.

In [69]:
threshold = np.linspace(0, 1, 100)

In [ ]:
columns_score_name = ['Threshold', 'Accuracy', 'Precision P', 'Recall P', 'F1 score P', \
                                              'Precision N', 'Recall N', 'F1 score N']
threshold_score = pd.concat([pd.DataFrame([compute_all_score(compute_confusion_matrix(test_label, prediction_proba, t ),t)]\
                                             , columns=columns_score_name) for t in threshold], ignore_index=True)
threshold_score.set_index('Threshold', inplace=True)

### D) Plot in a bar chart the coefficients of the logistic regression sorted by their contribution to the prediction.

In [70]:
logistic = LogisticRegression(solver='lbfgs', max_iter=10000)
logistic.fit(train_features, train_label)

LogisticRegression(max_iter=10000)

In [76]:
logistic.coef_[0]

dict = []

for name, value in zip(train_features, logistic.coef_[0]):
    dict.append({'name': name, 'coeff': value})

features_coef = pd.DataFrame(dict).sort_values('coeff')
features_coef.head()

,name,coeff
1,age_upon_intake_(years),-1.806586
6,sex_upon_outcome_Intact Male,-0.775349
5,sex_upon_outcome_Intact Female,-0.678463
26,intake_type_Wildlife,-0.663205
31,sex_upon_intake_Unknown,-0.496888



## Question 1: Which of the following metrics is most suitable when you are dealing with unbalanced classes?

- a) F1 Score
- b) Recall
- c) Precision
- d) Accuracy

In [77]:
'a'

'a'

## Question 2: You are working on a binary classification problem. You trained a model on a training dataset and got the following confusion matrix on the test dataset. What is true about the evaluation metrics (rounded to the second decimal point):

|            | Pred = NO|Pred=YES|
|------------|----------|--------|
| Actual NO  |    50    |   10   |
| Actual YES |    5     |   100  |

- a) Accuracy is 0.95
- b) Accuracy is 0.85
- c) False positive rate is 0.95
- d) True positive rate is 0.95

In [78]:
'd'

'd'